# 09 - Results Analysis

Este notebook sintetiza os principais resultados obtidos até agora:
- escolha da segmentação;
- baselines finais;
- importance de features;
- seleção de features para utilidade;
- trade-off entre utilidade e linkability;
- robustness da linkability;
- transformações de privacidade.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import FEATURE_SETS_DIR, FINAL_SEGMENT_FEATURES_DIR, OUTPUTS_TABLES_DIR
from modeling import compute_linkability_feature_importance, compute_utility_feature_importance

## Final Dataset Summary

In [2]:
manifest = json.loads((FINAL_SEGMENT_FEATURES_DIR / "manifest.json").read_text(encoding="utf-8"))
errors_path = FINAL_SEGMENT_FEATURES_DIR / "errors.csv"
errors_df = pd.read_csv(errors_path) if errors_path.exists() and errors_path.stat().st_size > 0 else pd.DataFrame()

pd.DataFrame(
    {
        "metric": [
            "record_count",
            "window_sec",
            "step_sec",
            "total_segments",
            "error_count",
            "chunk_count",
        ],
        "value": [
            manifest["record_count"],
            manifest["window_sec"],
            manifest["step_sec"],
            manifest["total_segments"],
            manifest.get("error_count", len(errors_df)),
            len(manifest["chunks"]),
        ],
    }
)

,metric,value
0,record_count,45152.0
1,window_sec,2.0
2,step_sec,1.0
3,total_segments,406359.0
4,error_count,1.0
5,chunk_count,181.0


## Segmentation Selection Summary

Resumo dos dois candidatos finais avaliados no estudo de segmentação.

In [3]:
segmentation_selection_df = pd.DataFrame(
    [
        {
            "config": "w2_o0p5",
            "window_sec": 2.0,
            "step_sec": 1.0,
            "utility_logreg_f1": 0.723567,
            "utility_logreg_balanced_accuracy": 0.854232,
            "utility_xgb_f1": 0.723949,
            "utility_xgb_roc_auc": 0.939952,
            "linkability_logreg_roc_auc": 0.996212,
            "linkability_xgb_roc_auc": 0.998898,
            "decision": "selected_main",
        },
        {
            "config": "w3_o0p5",
            "window_sec": 3.0,
            "step_sec": 1.5,
            "utility_logreg_f1": 0.737304,
            "utility_logreg_balanced_accuracy": 0.863631,
            "utility_xgb_f1": 0.754435,
            "utility_xgb_roc_auc": 0.950988,
            "linkability_logreg_roc_auc": 0.999280,
            "linkability_xgb_roc_auc": 0.999357,
            "decision": "higher_utility_comparison",
        },
    ]
)
segmentation_selection_df

,config,window_sec,step_sec,utility_logreg_f1,utility_logreg_balanced_accuracy,utility_xgb_f1,utility_xgb_roc_auc,linkability_logreg_roc_auc,linkability_xgb_roc_auc,decision
0,w2_o0p5,2.0,1.0,0.723567,0.854232,0.723949,0.939952,0.996212,0.998898,selected_main
1,w3_o0p5,3.0,1.5,0.737304,0.863631,0.754435,0.950988,0.999280,0.999357,higher_utility_comparison


## Final Baselines

Resultados finais das baselines no dataset final selecionado.

In [4]:
baseline_results_df = pd.DataFrame(
    [
        {
            "task": "utility",
            "model": "LogisticRegression",
            "f1_score": 0.678900,
            "balanced_accuracy": 0.831770,
            "roc_auc": 0.907671,
            "pr_auc": 0.712424,
            "status": "selected_baseline",
        },
        {
            "task": "utility",
            "model": "XGBoost",
            "f1_score": 0.675260,
            "balanced_accuracy": 0.775427,
            "roc_auc": 0.925902,
            "pr_auc": 0.779992,
            "status": "comparison",
        },
        {
            "task": "linkability",
            "model": "LogisticRegression",
            "f1_score": 0.975219,
            "balanced_accuracy": None,
            "roc_auc": 0.996190,
            "pr_auc": 0.996639,
            "status": "comparison",
        },
        {
            "task": "linkability",
            "model": "XGBoost",
            "f1_score": 0.978809,
            "balanced_accuracy": None,
            "roc_auc": 0.998343,
            "pr_auc": 0.998426,
            "status": "selected_baseline",
        },
    ]
)
baseline_results_df

,task,model,f1_score,balanced_accuracy,roc_auc,pr_auc,status
0,utility,LogisticRegression,0.678900,0.831770,0.907671,0.712424,selected_baseline
1,utility,XGBoost,0.675260,0.775427,0.925902,0.779992,comparison
2,linkability,LogisticRegression,0.975219,NaN,0.996190,0.996639,comparison
3,linkability,XGBoost,0.978809,NaN,0.998343,0.998426,selected_baseline


## Utility Feature Selection

Resumo dos experimentos `top-k` e da escolha do subset `top-150`.

In [5]:
utility_feature_selection_summary = pd.read_csv(FEATURE_SETS_DIR / "utility_feature_selection_summary.csv")
utility_feature_selection_summary

,ranking_model,model,subset_size,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc,feature_columns
0,LogisticRegression,LogisticRegression,10,0.358614,0.537019,0.569695,0.602199,0.286035,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
1,LogisticRegression,LogisticRegression,20,0.373723,0.538502,0.580740,0.616216,0.293602,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
2,LogisticRegression,LogisticRegression,30,0.381671,0.542876,0.587887,0.627807,0.308106,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
3,LogisticRegression,LogisticRegression,50,0.545827,0.678236,0.734271,0.810247,0.545013,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
4,LogisticRegression,LogisticRegression,75,0.625269,0.740641,0.793587,0.873484,0.652413,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
5,LogisticRegression,LogisticRegression,100,0.658877,0.765480,0.817777,0.896568,0.696704,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
6,LogisticRegression,LogisticRegression,150,0.678075,0.779402,0.831441,0.906544,0.710191,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
7,LogisticRegression,LogisticRegression,208,0.678864,0.780043,0.831829,0.907604,0.712167,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
8,LogisticRegression,XGBoost,10,0.031821,0.454777,0.506417,0.641087,0.333078,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."
9,LogisticRegression,XGBoost,20,0.057815,0.468000,0.512348,0.653330,0.350210,"['global_max_energy', 'lead_V5_std', 'lead_V1_..."


## Full 208 vs Utility Top-150

Comparação usando exatamente os mesmos segmentos para utilidade e linkability.

In [6]:
same_data_tradeoff_df = pd.DataFrame(
    [
        {
            "feature_set": "full_208",
            "utility_f1": 0.678900,
            "utility_balanced_accuracy": 0.831770,
            "utility_roc_auc": 0.907671,
            "linkability_logreg_roc_auc": 0.996190,
            "linkability_xgb_roc_auc": 0.998343,
        },
        {
            "feature_set": "utility_top150",
            "utility_f1": 0.678106,
            "utility_balanced_accuracy": 0.831456,
            "utility_roc_auc": 0.906530,
            "linkability_logreg_roc_auc": 0.993848,
            "linkability_xgb_roc_auc": 0.997051,
        },
    ]
)
same_data_tradeoff_df

,feature_set,utility_f1,utility_balanced_accuracy,utility_roc_auc,linkability_logreg_roc_auc,linkability_xgb_roc_auc
0,full_208,0.678900,0.831770,0.907671,0.996190,0.998343
1,utility_top150,0.678106,0.831456,0.906530,0.993848,0.997051


## Progressive Feature Removal from Top-150

Remoção das features mais importantes para linkability a partir do subset `top-150`.

In [7]:
feature_removal_tradeoff_df = pd.DataFrame(
    [
        {"n_removed": 0, "utility_f1": 0.678106, "utility_balanced_accuracy": 0.831456, "linkability_roc_auc": 0.997051},
        {"n_removed": 10, "utility_f1": 0.676987, "utility_balanced_accuracy": 0.830770, "linkability_roc_auc": 0.996015},
        {"n_removed": 20, "utility_f1": 0.665033, "utility_balanced_accuracy": 0.823306, "linkability_roc_auc": 0.995232},
        {"n_removed": 30, "utility_f1": 0.653883, "utility_balanced_accuracy": 0.815811, "linkability_roc_auc": 0.994811},
        {"n_removed": 40, "utility_f1": 0.646042, "utility_balanced_accuracy": 0.810015, "linkability_roc_auc": 0.993679},
        {"n_removed": 50, "utility_f1": 0.635907, "utility_balanced_accuracy": 0.802330, "linkability_roc_auc": 0.993234},
    ]
)
feature_removal_tradeoff_df

,n_removed,utility_f1,utility_balanced_accuracy,linkability_roc_auc
0,0,0.678106,0.831456,0.997051
1,10,0.676987,0.830770,0.996015
2,20,0.665033,0.823306,0.995232
3,30,0.653883,0.815811,0.994811
4,40,0.646042,0.810015,0.993679
5,50,0.635907,0.802330,0.993234


## Linkability Robustness Summary

Resumo dos checks de robustez com distância temporal, limitação de pares por paciente, baselines simples e múltiplas seeds.

In [8]:
linkability_robustness_df = pd.DataFrame(
    [
        {"setting": "standard", "model": "LogisticRegression", "f1_score": 0.982728, "roc_auc": 0.998161},
        {"setting": "standard", "model": "XGBoost", "f1_score": 0.983903, "roc_auc": 0.999101},
        {"setting": "harder", "model": "CosineSimilarity", "f1_score": 0.829690, "roc_auc": 0.892479},
        {"setting": "harder", "model": "EuclideanDistance", "f1_score": 0.000000, "roc_auc": 0.812707},
        {"setting": "harder", "model": "LogisticRegression", "f1_score": 0.982482, "roc_auc": 0.996604},
        {"setting": "harder", "model": "XGBoost", "f1_score": 0.983229, "roc_auc": 0.998676},
        {"setting": "repeated_mean", "model": "CosineSimilarity", "f1_score": 0.818678, "roc_auc": 0.884830},
        {"setting": "repeated_mean", "model": "EuclideanDistance", "f1_score": 0.000000, "roc_auc": 0.804087},
        {"setting": "repeated_mean", "model": "LogisticRegression", "f1_score": 0.981341, "roc_auc": 0.996924},
        {"setting": "repeated_mean", "model": "XGBoost", "f1_score": 0.983195, "roc_auc": 0.998672},
    ]
)
linkability_robustness_df

,setting,model,f1_score,roc_auc
0,standard,LogisticRegression,0.982728,0.998161
1,standard,XGBoost,0.983903,0.999101
2,harder,CosineSimilarity,0.829690,0.892479
3,harder,EuclideanDistance,0.000000,0.812707
4,harder,LogisticRegression,0.982482,0.996604
5,harder,XGBoost,0.983229,0.998676
6,repeated_mean,CosineSimilarity,0.818678,0.884830
7,repeated_mean,EuclideanDistance,0.000000,0.804087
8,repeated_mean,LogisticRegression,0.981341,0.996924
9,repeated_mean,XGBoost,0.983195,0.998672


## Privacy Transformations Summary

Resultados mais recentes das transformações de privacidade.

In [9]:
privacy_transformations_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transformations_summary.csv")
privacy_transformations_df

,transform_name,method,n_output_features,utility_f1,utility_balanced_accuracy,utility_roc_auc,utility_pr_auc,linkability_f1,linkability_roc_auc,linkability_pr_auc,delta_utility_f1,delta_utility_balanced_accuracy,delta_linkability_roc_auc
0,identity,identity,208,0.723567,0.854232,0.930469,0.794538,0.983976,0.998381,0.998482,0.000000,0.000000,0.000000
1,winsor_01_99,winsorization,208,0.729752,0.857262,0.931998,0.798535,0.983237,0.998797,0.998850,0.006184,0.003030,0.000416
2,winsor_05_95,winsorization,208,0.717855,0.848457,0.929945,0.798970,0.983483,0.998587,0.998635,-0.005712,-0.005775,0.000206
3,winsor_10_90,winsorization,208,0.721771,0.851521,0.929105,0.793556,0.983229,0.998367,0.998413,-0.001796,-0.002710,-0.000014
4,pca_120,pca,120,0.714832,0.847961,0.927374,0.790344,0.929126,0.982065,0.983659,-0.008735,-0.006271,-0.016316


## Feature Importance Refresh

Esta secção mantém a análise de importance no dataset final selecionado. Se necessário, limita `MAX_CHUNKS` para uma corrida mais leve.

In [10]:
MAX_CHUNKS = 20  # set None for the full dataset if needed

chunk_files = [FINAL_SEGMENT_FEATURES_DIR / chunk["chunk_file"] for chunk in manifest["chunks"]]
if MAX_CHUNKS is not None:
    chunk_files = chunk_files[:MAX_CHUNKS]

features_df = pd.concat(
    [pd.read_csv(chunk_file, compression="gzip", low_memory=False) for chunk_file in chunk_files],
    ignore_index=True,
)

print("Chunk files loaded for importance:", len(chunk_files))
print("Features dataframe:", features_df.shape)

Chunk files loaded for importance: 20
Features dataframe: (45000, 215)


In [11]:
utility_importance = compute_utility_feature_importance(
    features_df=features_df,
    model_name="LogisticRegression",
    test_size=0.2,
    random_state=42,
    permutation_scoring="average_precision",
)

utility_importance["model_based_df"].head(15)

,feature,importance
0,lead_V6_std,2.790300
1,lead_V6_rms,2.347862
2,global_max_energy,2.264587
3,lead_V5_std,1.810886
4,lead_V4_std,1.804211
5,global_max_std,1.689254
6,global_std_energy,1.669288
7,lead_V4_rms,1.666001
8,lead_V5_rms,1.633625
9,global_std_mean,1.538205


In [12]:
linkability_importance = compute_linkability_feature_importance(
    features_df=features_df,
    model_name="LogisticRegression",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
    max_positive_pairs_per_patient=3,
    permutation_scoring="roc_auc",
)

linkability_importance["model_based_df"].head(15)

,feature,importance
0,absdiff__lead_V6_max,2.061130
1,absdiff__lead_V4_min,1.730442
2,absdiff__global_mean_amplitude,1.708739
3,absdiff__lead_V5_min,1.622317
4,absdiff__lead_V3_max,1.611434
5,absdiff__global_min_energy,1.273390
6,absdiff__lead_V1_min,1.266173
7,absdiff__global_mean_kurtosis,1.213684
8,absdiff__global_std_zero_crossing_rate,1.208659
9,absdiff__global_mean_energy,1.150682


In [13]:
utility_top = utility_importance["model_based_df"].head(20).copy()
utility_top["task"] = "utility"

linkability_top = linkability_importance["model_based_df"].head(20).copy()
linkability_top["task"] = "linkability"

comparison_df = pd.concat([utility_top, linkability_top], ignore_index=True)
comparison_df

,feature,importance,task
0,lead_V6_std,2.790300,utility
1,lead_V6_rms,2.347862,utility
2,global_max_energy,2.264587,utility
3,lead_V5_std,1.810886,utility
4,lead_V4_std,1.804211,utility
5,global_max_std,1.689254,utility
6,global_std_energy,1.669288,utility
7,lead_V4_rms,1.666001,utility
8,lead_V5_rms,1.633625,utility
9,global_std_mean,1.538205,utility


In [14]:
utility_feature_set = set(utility_importance["model_based_df"].head(20)["feature"])
linkability_feature_set = set(linkability_importance["model_based_df"].head(20)["feature"])

print("Overlap in top-20 model-based features:", len(utility_feature_set & linkability_feature_set))
sorted(utility_feature_set & linkability_feature_set)

Overlap in top-20 model-based features: 0


[]

## Current Takeaways

- `w2_o0p5` foi a configuração de segmentação principal escolhida.
- A baseline principal de utilidade é `LogisticRegression`.
- A baseline principal de linkability é `XGBoost`.
- `utility_top150` preserva quase toda a utilidade, mas reduz pouco a linkability.
- Remoção simples de features não reduz a linkability de forma forte.
- A linkability mantém-se muito alta mesmo com checks de robustez.
- Entre as transformações de privacidade testadas, `PCA` foi a mais promissora.
- O melhor candidato até agora é a comparação `identity` vs a melhor variante atual em `privacy_transformations_summary.csv`.